<a href="https://colab.research.google.com/github/Anil-Pradhan-web/touch-grass-nature-advisor/blob/main/touch_grass_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate torch pydantic

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login

login("YOUR_HF_TOKEN")

In [ ]:
import json
import math
import torch
from typing import Dict, Any
from datetime import datetime
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM

class OutdoorPlanSchema(BaseModel):
    objective: str = Field(description="Summary of the screen-free outdoor activity")
    preparation_checklist: list[str] = Field(description="Crucial gear/supplies needed based on telemetry")
    safety_advisory: str = Field(description="Actionable precautions for terrain and sunlight conditions")
    screen_free_action: str = Field(description="Tactile task designed to eliminate phone interaction")
    mindfulness_anchor: str = Field(description="Specific sensory observation target")

class OfflineEnvironmentEngine:
    @staticmethod
    def calculate_sun_elevation(lat: float, day_of_year: int, hour_utc: float) -> float:
        declination = 23.45 * math.sin(math.radians((360 / 365) * (day_of_year - 81)))
        hour_angle = 15.0 * (hour_utc - 12.0)
        elevation = math.asin(
            math.sin(math.radians(lat)) * math.sin(math.radians(declination)) +
            math.cos(math.radians(lat)) * math.cos(math.radians(declination)) * math.cos(math.radians(hour_angle))
        )
        return round(math.degrees(elevation), 2)

    @classmethod
    def analyze_trail_telemetry(cls, lat: float, elevation_m: int, reported_temp_c: float) -> Dict[str, Any]:
        day_of_year = datetime.now().timetuple().tm_yday
        hour_utc = 10.0
        solar_angle = cls.calculate_sun_elevation(lat, day_of_year, hour_utc)
        apparent_temp = reported_temp_c - (elevation_m / 1000.0 * 6.5)

        return {
            "solar_elevation_deg": solar_angle,
            "estimated_summit_temp_c": round(apparent_temp, 1),
            "uv_exposure_risk": "High" if solar_angle > 45 else "Moderate" if solar_angle > 20 else "Low",
            "daylight_status": "Adequate" if solar_angle > 10 else "Golden Hour / Twilight Risk"
        }

class TouchGrassAgent:
    def __init__(self, model_id: str = "google/gemma-2-2b-it"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=torch.bfloat16,
            device_map="auto"
        )
        self.env_engine = OfflineEnvironmentEngine()

    def generate_plan(self, activity: str, lat: float, elevation_m: int, temp_c: float) -> OutdoorPlanSchema:
        telemetry = self.env_engine.analyze_trail_telemetry(lat, elevation_m, temp_c)

        system_instruction = (
            "You are an offline edge agent built to disconnect engineers from screens and guide outdoor expeditions. "
            "You strictly enforce structured output format. Return raw, valid JSON only matching this schema:\n"
            "{\n"
            '  "objective": "...",\n'
            '  "preparation_checklist": ["item 1", "item 2", "item 3"],\n'
            '  "safety_advisory": "...",\n'
            '  "screen_free_action": "...",\n'
            '  "mindfulness_anchor": "..."\n'
            "}"
        )

        user_prompt = (
            f"Activity: {activity}\n"
            f"Sensor Telemetry: {telemetry}\n"
            "Generate an expedition itinerary tailored to keep the user entirely off mobile screens."
        )

        messages = [
            {"role": "user", "content": f"{system_instruction}\n\n{user_prompt}"}
        ]

        model_inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        )

        model_inputs = {k: v.to(self.device) for k, v in model_inputs.items()}
        prompt_len = model_inputs["input_ids"].shape[-1]

        with torch.no_grad():
            output_tokens = self.model.generate(
                **model_inputs,
                max_new_tokens=400,
                temperature=0.3,
                top_p=0.9,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = output_tokens[0][prompt_len:]
        raw_output = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        try:
            cleaned_json = raw_output.replace("```json", "").replace("```", "").strip()
            data = json.loads(cleaned_json)
            return OutdoorPlanSchema(**data)
        except Exception:
            return OutdoorPlanSchema(
                objective=f"Screen-free {activity}",
                preparation_checklist=["Water bladder", "Physical trail map", "Hydration pack"],
                safety_advisory=f"Telemetry calculated: Summit Temp {telemetry['estimated_summit_temp_c']}°C, UV: {telemetry['uv_exposure_risk']}",
                screen_free_action="Stow device inside backpack in power-off state for minimum 90 minutes.",
                mindfulness_anchor="Audit native leaf textures and monitor ambient wind cadence."
            )

In [ ]:
agent = TouchGrassAgent()

result = agent.generate_plan(
    activity="Alpine trail hike and botany exploration",
    lat=20.2961,
    elevation_m=850,
    temp_c=26.0
)

print(json.dumps(result.model_dump(), indent=2))

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

{
  "objective": "Enjoy a scenic alpine trail hike and botanical exploration, immersing yourself in nature and appreciating the local flora.",
  "preparation_checklist": [
    "Pack a sturdy backpack with snacks, water, and a first-aid kit.",
    "Wear appropriate hiking boots and clothing for varying weather conditions.",
    "Bring a field guide for identifying local plants."
  ],
  "safety_advisory": "Stay on marked trails, be aware of your surroundings, and inform someone of your hiking plans. Carry a whistle for emergencies and watch out for wildlife.",
  "screen_free_action": "Engage in mindful observation of the natural environment. Focus on the sounds, sights, and textures around you. Identify different plant species, noting their unique features and growth patterns.",
  "mindfulness_anchor": "Take deep breaths and appreciate the beauty of the natural world. Focus on the present moment and let go of any distractions."
}
